In [ ]:
import os

REPO_URL = "https://github.com/meriem200512365/Chat-boot-cegedim.git"
REPO_DIR = "Chat-boot-cegedim"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}

In [ ]:
import json

with open("data/chemins/menu_index.json", encoding="utf-8") as f:
    knowledge_base = json.load(f)

corpus_texts = [item["search_text"] for item in knowledge_base]
corpus_ids = [item["id"] for item in knowledge_base]
print(f"{len(knowledge_base)} chemins charges.")
print("Exemple :", knowledge_base[0])

In [ ]:
# A adapter avec des vrais libelles observes ci-dessus si besoin.
# Format : (question posee, id du chemin attendu)
QUESTIONS_TEST = [
    ("je veux gerer les actes RO", None),
    ("comment parametrer un devis web", None),
    ("ou trouver la gestion des cheques", None),
    ("annuler un cheque", None),
    ("acces aux referentiels de prestations", None),
    ("configuration du parametrage general", None),
]

# Astuce : pour completer les id attendus, cherche dans knowledge_base
# les chemins correspondant a chaque question (recherche par mot-cle).
def chercher_id_par_mot_cle(mot):
    return [(it["id"], it["path_str"]) for it in knowledge_base if mot.lower() in it["search_text"].lower()]

for mot in ["Actes", "Devis", "Cheque", "Parametrage"]:
    print(f"\n--- '{mot}' ---")
    for id_, path in chercher_id_par_mot_cle(mot)[:5]:
        print(" ", id_, "->", path)


In [ ]:
MODELES = [
    "paraphrase-multilingual-MiniLM-L12-v2",   # modele actuel du projet (leger, ~470 Mo)
    "distiluse-base-multilingual-cased-v2",    # alternative legere
    "paraphrase-multilingual-mpnet-base-v2",   # plus lourd, potentiellement plus precis
]


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import time

resultats_par_modele = {}

for nom_modele in MODELES:
    print(f"\n=== Chargement de {nom_modele} ===")
    t0 = time.time()
    model = SentenceTransformer(nom_modele)
    print(f"Charge en {time.time()-t0:.1f}s")

    t0 = time.time()
    corpus_emb = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)
    print(f"Corpus encode en {time.time()-t0:.1f}s")

    lignes = []
    for question, expected_id in QUESTIONS_TEST:
        q_emb = model.encode(question, convert_to_tensor=True)
        cos_scores = util.cos_sim(q_emb, corpus_emb)[0]
        distances = 1 - cos_scores.cpu().numpy()  # distance cosinus = 1 - similarite
        top_idx = int(np.argmin(distances))

        lignes.append({
            "modele": nom_modele,
            "question": question,
            "id_trouve": corpus_ids[top_idx],
            "path_trouve": knowledge_base[top_idx]["path_str"],
            "distance": round(float(distances[top_idx]), 4),
            "id_attendu": expected_id,
            "correct": (expected_id is not None and corpus_ids[top_idx] == expected_id),
        })

    resultats_par_modele[nom_modele] = lignes


In [ ]:
import pandas as pd

frames = []
for nom_modele, lignes in resultats_par_modele.items():
    frames.append(pd.DataFrame(lignes))
df_comparaison = pd.concat(frames, ignore_index=True)
df_comparaison[["modele", "question", "path_trouve", "distance", "correct"]]


In [ ]:
import matplotlib.pyplot as plt

modele_actuel = "paraphrase-multilingual-MiniLM-L12-v2"
distances_actuel = [l["distance"] for l in resultats_par_modele[modele_actuel]]

plt.figure(figsize=(8, 4))
plt.hist(distances_actuel, bins=10, edgecolor="black")
plt.axvline(0.30, color="green", linestyle="--", label="SEUIL_CONFIANT = 0.30")
plt.axvline(0.55, color="red", linestyle="--", label="SEUIL_INCERTAIN = 0.55")
plt.title(f"Distribution des distances du meilleur resultat ({modele_actuel})")
plt.xlabel("Distance cosinus")
plt.ylabel("Nombre de questions")
plt.legend()
plt.tight_layout()
plt.show()
